# Member 2: Markov Decision Process Construction and Policy Iteration

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append('..')
from member_2_rl.mdp_builder import MDPBuilder
from member_2_rl.policy_iteration import PolicyIteration
from member_2_rl.td_evaluator import TemporalDifferenceEvaluator
from member_2_rl.rl_trainer import RLTrainer

os.makedirs('../artifacts', exist_ok=True)


### 1. Load Training Trajectories

In [ ]:
tr_train = pd.read_csv('../data/trajectories_train.csv')
print("Total transition records:", len(tr_train))
print("Unique states observed:", tr_train["state"].nunique())
print("Unique actions observed:", tr_train["action"].nunique())
tr_train.head(6)


### 2. Build MDP Transition and Reward Matrices (with Safety Masking)

In [ ]:
mdp = MDPBuilder(n_states=75, n_actions=25, min_action_count=5)
mdp.build_from_trajectories(tr_train)

print("Transition matrix shape:", mdp.transitions.shape)
print("Reward matrix shape:    ", mdp.rewards.shape)
print("Valid action mask shape:", mdp.valid_action_mask.shape)
print("Average supported actions per state:", round(float(np.mean(mdp.valid_action_mask[:75].sum(axis=1))), 2))


### 3. Policy Iteration Training

In [ ]:
pi_solver = PolicyIteration(mdp=mdp, gamma=0.99, theta=1e-5, max_pi_iters=50)
pi_solver.fit()

history_df = pd.DataFrame(pi_solver.history)
print("Converged in iterations:", len(history_df))
history_df


### 4. TD Learning Baseline for Clinician Policy

In [ ]:
td_solver = TemporalDifferenceEvaluator(n_states=75, n_actions=25, gamma=0.99, alpha=0.01)
td_solver.fit(tr_train)
td_v = td_solver.get_clinician_state_values()

print(f"Mean AI Policy State Value V(s):        {float(np.mean(pi_solver.V[:75])):.2f}")
print(f"Mean Clinician Policy State Value V(s): {float(np.mean(td_v)):.2f}")


### 5. Convergence Curve Visualization

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4.5), dpi=150)

ax1.plot(history_df['iteration'], history_df['mean_v'], color='#1B4F72', marker='o', linewidth=2, label='Mean V(s)')
ax1.set_xlabel('Policy Iteration Step', fontsize=11)
ax1.set_ylabel('Mean State Value V(s)', color='#1B4F72', fontsize=11)
ax1.tick_params(axis='y', labelcolor='#1B4F72')
ax1.grid(True, linestyle='--', alpha=0.4)

ax2 = ax1.twinx()
ax2.bar(history_df['iteration'], history_df['actions_changed'], color='#E74C3C', alpha=0.35, width=0.4, label='Policy Changes')
ax2.set_ylabel('Number of Policy Changes', color='#E74C3C', fontsize=11)
ax2.tick_params(axis='y', labelcolor='#E74C3C')

plt.title('Policy Iteration Convergence on Sepsis MDP', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.show()


### 6. Inspect Learned Optimal Policy vs Human Clinician Choices

In [ ]:
ai_policy = pi_solver.policy[:75]
clinician_mode_actions = tr_train.groupby('state')['action'].agg(lambda x: x.mode()[0]).to_dict()

comparison = []
for s in range(15):
    c_act = clinician_mode_actions.get(s, -1)
    a_act = ai_policy[s]
    comparison.append({
        'state': s,
        'clinician_action': c_act,
        'clin_fluid_bin': c_act // 5 if c_act >= 0 else None,
        'clin_vaso_bin': c_act % 5 if c_act >= 0 else None,
        'ai_action': a_act,
        'ai_fluid_bin': a_act // 5,
        'ai_vaso_bin': a_act % 5,
        'v_value': round(float(pi_solver.V[s]), 2)
    })

pd.DataFrame(comparison)


### 7. Export Trained Policy Artifacts

In [ ]:
trainer = RLTrainer(n_states=75, n_actions=25, min_action_count=5, gamma=0.99)
trainer.mdp = mdp
trainer.pi = pi_solver
trainer.td_eval = td_solver
trainer.export('../artifacts/ai_policy.json', '../artifacts/mdp_matrices.npz')
print("Exported AI policy and MDP matrices to artifacts.")
